In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [3]:
# Let's read all files in the "logs" folder and sort them by creation time (most recent last).
log_folder = "logs"
log_files = [os.path.join(log_folder, f) for f in os.listdir(log_folder) if os.path.isfile(os.path.join(log_folder, f))]
log_files_sorted = sorted(log_files, key=lambda x: os.path.getctime(x))


In [8]:
log_df = pd.read_csv(
    log_files_sorted[-1],
    names=["time", "module", "level", "message", "function", "further"]
)

In [11]:
# Convert the "time" column to datetime objects for easier time-based analysis.
log_df["time"] = pd.to_datetime(log_df["time"])


In [16]:
log_df

,time,module,level,message,function,further
0,2025-09-30 14:50:40.748,ovc,INFO,user recording started,record_user,1 seconds
1,2025-09-30 14:50:56.676,ovc,INFO,speech detected,record_user,NaN
2,2025-09-30 14:50:57.664,ovc,INFO,user recording ended,record_user,NaN
3,2025-09-30 14:50:57.689,ovc,INFO,transcribing,STT,NaN
4,2025-09-30 14:50:58.060,ovc,INFO,transcribed,STT,Hello.
...,...,...,...,...,...,...
188,2025-09-30 14:52:46.356,ovc,INFO,running tts,TTS,Have a great day!
189,2025-09-30 14:52:46.748,ovc,INFO,tts output received,TTS,NaN
190,2025-09-30 14:52:47.112,ovc,INFO,playing audio,TTS,1.65 seconds
191,2025-09-30 14:52:51.742,ovc,INFO,audio play thread ended,TTS,NaN


### What do we want to know?

- How much time it took to transcribe? 
- How much time it took to get the first token? 
- How much time it took to synthesize? 
- How much time it took to the first audio?

In [24]:
log_df.head(20)

,time,module,level,message,function,further
0,2025-09-30 14:50:40.748,ovc,INFO,user recording started,record_user,1 seconds
1,2025-09-30 14:50:56.676,ovc,INFO,speech detected,record_user,NaN
2,2025-09-30 14:50:57.664,ovc,INFO,user recording ended,record_user,NaN
3,2025-09-30 14:50:57.689,ovc,INFO,transcribing,STT,NaN
4,2025-09-30 14:50:58.060,ovc,INFO,transcribed,STT,Hello.
5,2025-09-30 14:50:58.066,ovc,INFO,segmenting,STT,Hello.
6,2025-09-30 14:50:58.078,ovc,INFO,sentence boundary detected,STT,Hello.
7,2025-09-30 14:50:58.102,ovc,INFO,audio play thread started,TTS,NaN
8,2025-09-30 14:50:58.126,ovc,INFO,getting all text,TTS,NaN
9,2025-09-30 14:51:09.615,ovc,INFO,all text received,TTS,NaN


In [21]:
log_df["message"].unique()

array(['user recording started', 'speech detected',
       'user recording ended', 'transcribing', 'transcribed',
       'segmenting', 'sentence boundary detected',
       'audio play thread started', 'getting all text',
       'all text received', 'segmenting text', 'single sentence detected',
       'multiple sentences detected', 'cleaning sentence', 'running tts',
       'tts output received', 'playing audio', 'Stream ended',
       'audio play thread ended'], dtype=object)

In [ ]:
log_df[log_df["message"] == "transcribing"]

,time,module,level,message,function,further
3,2025-09-30 14:50:57.689,ovc,INFO,transcribing,STT,NaN
39,2025-09-30 14:51:21.017,ovc,INFO,transcribing,STT,NaN
67,2025-09-30 14:51:36.717,ovc,INFO,transcribing,STT,NaN
123,2025-09-30 14:52:03.779,ovc,INFO,transcribing,STT,NaN
159,2025-09-30 14:52:40.856,ovc,INFO,transcribing,STT,NaN


In [19]:
log_df[log_df["message"] == "transcribed"]

,time,module,level,message,function,further
4,2025-09-30 14:50:58.060,ovc,INFO,transcribed,STT,Hello.
40,2025-09-30 14:51:21.927,ovc,INFO,transcribed,STT,I am looking to buy a new iPhone.
68,2025-09-30 14:51:37.455,ovc,INFO,transcribed,STT,"Yeah, I'm looking for something with a good ca..."
124,2025-09-30 14:52:04.190,ovc,INFO,transcribed,STT,Yes.
160,2025-09-30 14:52:41.411,ovc,INFO,transcribed,STT,"No, I think that's it. Thank you."


In [27]:
transcribe_time = log_df[log_df["message"] == "transcribed"]["time"].values - log_df[log_df["message"] == "transcribing"]["time"].values
transcribe_time






array([371000000, 910000000, 738000000, 411000000, 555000000],
      dtype='timedelta64[ns]')

In [ ]:
transcribe_time


array([371000000000, 910000000000, 738000000000, 411000000000,
       555000000000], dtype='timedelta64[ns]')